# Soccer tracking on Colab — YOLO + ByteTrack (no gating, no keys)

Role A / Phase 1. **Runtime > Change runtime type > T4 GPU**, then Run all.

Why YOLO and not SAM 3.1: SAM 3's weights are gated on Hugging Face (slow/uncertain
approval) and require a heavier GPU runtime — too much friction for a 24h
deadline. Ultralytics YOLO is **ungated, free, auto-downloads its weights, and runs on
a free T4 in seconds**, with built-in multi-object tracking (persistent IDs). It plugs
into the exact same `SamBackend` interface, so the rest of the pipeline (minimap, HOTA
eval) doesn't care which tracker produced the boxes. SAM 3.1 stays wired up
(`sam.backend: local`/`api`) for if/when access clears — see `docs/COMPUTE.md`.

Note: YOLO detects COCO classes, so players/keeper/referee all come back as `person`
(team/role split comes later via jersey-color clustering — `docs/ML_DIRECTIONS.md`).
The ball is COCO `sports ball`.

In [ ]:
# 1. Confirm the GPU runtime
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone + install (no gated weights, no HF token, no runtime restart needed)
import os

REPO_URL = "https://github.com/wheredawoodat949/AI-Hackathon"
BRANCH = "feat/tracking-ashmeet"

if not os.path.isdir("AI-Hackathon"):
    !git clone --branch {BRANCH} {REPO_URL}.git
%cd AI-Hackathon
!pip install -q -r requirements.txt
!pip install -q -U ultralytics
!python -m src.utils.gpu        # must print CUDA

In [ ]:
# 3. Get a real clip from the Drive mirror (no auth)
!python -m src.data.download --match 117093

In [ ]:
# 4. Run real tracking (yolo11n weights auto-download on first use) and render a clip.
import glob

video = sorted(glob.glob("data/videos/117093*"))[0]
print("tracking:", video)
!python -m src.tracking.demo --backend yolo --video {video} --seconds 10

In [ ]:
# 5. Preview the annotated output (and download it from the Colab file browser).
import glob

out = sorted(glob.glob("outputs/track_117093_yolo*"))
print(out)

from IPython.display import Video
Video(out[0], embed=True, width=720) if out else print("no output found — check cell 4")

## If this worked
You have real tracked players on real footage. Download `outputs/track_117093_yolo.mp4`
(it's gitignored), drop it in the team Drive, and tell **Role B** (pitch+eval) that real
per-frame detections with track IDs are available via `sam.backend: yolo` — the
homography → minimap → HOTA work can run on real output now, not just the replay baseline.

## Accuracy knobs (if players are missed / boxes look off)
- Bump the model: set `sam.yolo_weights: yolo11s.pt` (or `yolo11m.pt`) in `config.yaml`.
- Lower the threshold: `sam.conf: 0.15` catches more distant players (panoramic = small).
- The 4K panorama is wide; YOLO downsamples it — for best results we may later tile the
  frame. Fine for a first real demo as-is.